<a href="https://colab.research.google.com/github/karywnl/kaggle/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.environ["KAGGLE_CONFIG_DIR"] = "/content/drive/MyDrive/Kaggle"

!chmod 600 "/content/drive/MyDrive/Kaggle/kaggle.json"

Mounted at /content/drive


In [3]:
!kaggle datasets download -d "maxhorowitz/nflplaybyplay2009to2016"
!unzip nflplaybyplay2009to2016.zip

Dataset URL: https://www.kaggle.com/datasets/maxhorowitz/nflplaybyplay2009to2016
License(s): unknown
 78% 215M/274M [00:00<00:00, 631MB/s] 
100% 274M/274M [00:00<00:00, 600MB/s]
Archive:  nflplaybyplay2009to2016.zip
  inflating: NFL Play by Play 2009-2016 (v3).csv  
  inflating: NFL Play by Play 2009-2017 (v4).csv  
  inflating: NFL Play by Play 2009-2018 (v5).csv  


In [4]:
nfl_data = pd.read_csv("NFL Play by Play 2009-2017 (v4).csv")

/tmp/ipykernel_491/2226537327.py:1: DtypeWarning: Columns (25,51) have mixed types. Specify dtype option on import or set low_memory=False.
  nfl_data = pd.read_csv("NFL Play by Play 2009-2017 (v4).csv")


So, there is a concept of low memory in pandas, when the dataset is huge. The pandas doesn't read them all at once. Instead it will read them in chunk. So, the problem with this approach is different chunks can be seen as a different datatype. That's why we get the mixed types warning in the above cell. If low_memory = False, then whole dataset is read at once and we may get no warning. Usually if the size of the dataset is like > 50 MB.

In [5]:
nfl_data = pd.read_csv("NFL Play by Play 2009-2017 (v4).csv", low_memory=False)
nfl_data.head()

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


# **Handling Missing Values**

In [6]:
nfl_data.isnull().sum()

,0
Date,0
GameID,0
Drive,0
qtr,0
down,61154
...,...
Win_Prob,25009
WPA,5541
airWPA,248501
yacWPA,248762


.sum() here is by default goes by column-wise because its axis=0 by default. If axis=1, then it goes row-wise.

*   axis = 0 : Null count by row - col b col and col on the index
*   axis = 1 : Null count by col - row by row and row on the index

In [7]:
nfl_data.isnull()

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,True,True,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,True,True,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407683,False,False,False,False,True,False,False,False,False,False,...,True,True,True,True,True,True,False,True,True,False
407684,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
407685,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
407686,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,True,True,False


nfl_data.isnull() creates a dataframe, with booleans in each cell, Indicating that whethere that cell is True or False.

nfl_data.isnull().sum() goes column-wise and tells how many null values each column has and more importantly it returns a Series.

Series have same property just like list and it can also be sliced. For eg : we could get select particulars and see how many null are there in those range of rows.

The below code demonstrates, how to select the null count from first 10 rows column-wise.

In [8]:
nfl_data.isnull().sum()[0:10]

,0
Date,0
GameID,0
Drive,0
qtr,0
down,61154
time,224
TimeUnder,0
TimeSecs,224
PlayTimeDiff,444
SideofField,528


In [9]:
nfl_data.isnull().sum()

,0
Date,0
GameID,0
Drive,0
qtr,0
down,61154
...,...
Win_Prob,25009
WPA,5541
airWPA,248501
yacWPA,248762


***Imputation : You make an educated guess of what the missing value would be and fill them, instead of just dropping them.***

In [10]:
nfl_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Columns: 102 entries, Date to Season
dtypes: float64(33), int64(31), object(38)
memory usage: 317.3+ MB


colab usually truncates the output if it is too long like in this case there are 102 columns. we need to use nfl_data.info(verbose=True) to output the whole output.

In [11]:
!kaggle datasets download -d "aparnashastry/building-permit-applications-data"
!unzip building-permit-applications-data.zip

Dataset URL: https://www.kaggle.com/datasets/aparnashastry/building-permit-applications-data
License(s): DbCL-1.0
  0% 0.00/18.0M [00:00<?, ?B/s]
100% 18.0M/18.0M [00:00<00:00, 1.17GB/s]
Archive:  building-permit-applications-data.zip
  inflating: Building_Permits.csv    
  inflating: DataDictionaryBuildingPermit.xlsx  


In [12]:
sf_permits = pd.read_csv("Building_Permits.csv")
sf_permits.head()

/tmp/ipykernel_491/192664614.py:1: DtypeWarning: Columns (22,32) have mixed types. Specify dtype option on import or set low_memory=False.
  sf_permits = pd.read_csv("Building_Permits.csv")


,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Existing Construction Type,Existing Construction Type Description,Proposed Construction Type,Proposed Construction Type Description,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID
0,201505065519,4,sign - erect,05/06/2015,0326,023,140,NaN,Ellis,St,...,3.0,constr type 3,NaN,NaN,NaN,3.0,Tenderloin,94102.0,"(37.785719256680785, -122.40852313194863)",1380611233945
1,201604195146,4,sign - erect,04/19/2016,0306,007,440,NaN,Geary,St,...,3.0,constr type 3,NaN,NaN,NaN,3.0,Tenderloin,94102.0,"(37.78733980600732, -122.41063199757738)",1420164406718
2,201605278609,3,additions alterations or repairs,05/27/2016,0595,203,1647,NaN,Pacific,Av,...,1.0,constr type 1,1.0,constr type 1,NaN,3.0,Russian Hill,94109.0,"(37.7946573324287, -122.42232562979227)",1424856504716
3,201611072166,8,otc alterations permit,11/07/2016,0156,011,1230,NaN,Pacific,Av,...,5.0,wood frame (5),5.0,wood frame (5),NaN,3.0,Nob Hill,94109.0,"(37.79595867909168, -122.41557405519474)",1443574295566
4,201611283529,6,demolitions,11/28/2016,0342,001,950,NaN,Market,St,...,3.0,constr type 3,NaN,NaN,NaN,6.0,Tenderloin,94102.0,"(37.78315261897309, -122.40950883997789)",144548169992


What percentage of the values in the dataset are missing? Your answer should be a number between 0 and 100. (If 1/4 of the values in the dataset are missing, the answer is 25.)

In [13]:
(sf_permits.isnull().sum().sum() / np.prod(sf_permits.shape)) * 100

np.float64(26.26002315058403)

If you removed all of the rows of sf_permits with missing values, how many rows are left?

In [14]:
sf_permits.dropna(axis=0).shape[0]

0

Now try removing all the columns with empty values.

*   Create a new DataFrame called sf_permits_with_na_dropped that has all of the columns with empty values removed.
*   How many columns were removed from the original sf_permits DataFrame? Use this number to set the value of the dropped_columns variable below.



In [15]:
sf_permits_na_dropped = sf_permits.dropna(axis=1)
dropped_columns = sf_permits.shape[1] - sf_permits_na_dropped.shape[1]

Try replacing all the NaN's in the sf_permits data with the one that comes directly after it and then replacing any remaining NaN's with 0. Set the result to a new DataFrame sf_permits_with_na_imputed.

In [16]:
sf_permits.fillna(method="bfill").fillna(0)

/tmp/ipykernel_491/3525226833.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sf_permits.fillna(method="bfill").fillna(0)


,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Existing Construction Type,Existing Construction Type Description,Proposed Construction Type,Proposed Construction Type Description,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID
0,201505065519,4,sign - erect,05/06/2015,0326,023,140,A,Ellis,St,...,3.0,constr type 3,1.0,constr type 1,Y,3.0,Tenderloin,94102.0,"(37.785719256680785, -122.40852313194863)",1380611233945
1,201604195146,4,sign - erect,04/19/2016,0306,007,440,A,Geary,St,...,3.0,constr type 3,1.0,constr type 1,Y,3.0,Tenderloin,94102.0,"(37.78733980600732, -122.41063199757738)",1420164406718
2,201605278609,3,additions alterations or repairs,05/27/2016,0595,203,1647,A,Pacific,Av,...,1.0,constr type 1,1.0,constr type 1,Y,3.0,Russian Hill,94109.0,"(37.7946573324287, -122.42232562979227)",1424856504716
3,201611072166,8,otc alterations permit,11/07/2016,0156,011,1230,A,Pacific,Av,...,5.0,wood frame (5),5.0,wood frame (5),Y,3.0,Nob Hill,94109.0,"(37.79595867909168, -122.41557405519474)",1443574295566
4,201611283529,6,demolitions,11/28/2016,0342,001,950,A,Market,St,...,3.0,constr type 3,1.0,constr type 1,Y,6.0,Tenderloin,94102.0,"(37.78315261897309, -122.40950883997789)",144548169992
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198895,M862628,8,otc alterations permit,12/05/2017,0113,017A,1228,0,Montgomery,St,...,5.0,wood frame (5),5.0,wood frame (5),0,0.0,0,0.0,0,1489337276729
198896,201712055595,8,otc alterations permit,12/05/2017,0271,014,580,0,Bush,St,...,5.0,wood frame (5),5.0,wood frame (5),0,0.0,0,0.0,0,1489462354993
198897,M863507,8,otc alterations permit,12/06/2017,4318,019,1568,0,Indiana,St,...,0.0,0,0.0,0,0,0.0,0,0.0,0,1489539379952
198898,M863747,8,otc alterations permit,12/06/2017,0298,029,795,0,Sutter,St,...,0.0,0,0.0,0,0,0.0,0,0.0,0,1489608233656


Here, we used .fillna() twice inorder to fill the values with 0, that are still NaN even after the first back fill. This might happen if the last row is NaN.

Some notable .fillna() methods are
*   bfill - Fills the value of next row
*   ffill - Fills the value of previous row

# **Scaling and Normalization**

In [17]:
!kaggle datasets download -d "kemical/kickstarter-projects"
!unzip kickstarter-projects.zip

Dataset URL: https://www.kaggle.com/datasets/kemical/kickstarter-projects
License(s): CC-BY-NC-SA-4.0
  0% 0.00/36.8M [00:00<?, ?B/s]
100% 36.8M/36.8M [00:00<00:00, 841MB/s]
Archive:  kickstarter-projects.zip
  inflating: ks-projects-201612.csv  
  inflating: ks-projects-201801.csv  


In [18]:
kickstarters = pd.read_csv("ks-projects-201801.csv", index_col="ID")
kickstarters.loc[:, ["goal", "pledged", "backers", "usd pledged"]]

,goal,pledged,backers,usd pledged
ID,,,,
1000002330,1000.0,0.0,0,0.0
1000003930,30000.0,2421.0,15,100.0
1000004038,45000.0,220.0,3,220.0
1000007540,5000.0,1.0,1,1.0
1000011046,19500.0,1283.0,14,1283.0
...,...,...,...,...
999976400,50000.0,25.0,1,25.0
999977640,1500.0,155.0,5,155.0
999986353,15000.0,20.0,1,20.0


In [19]:
kickstarters.loc[:, ["goal", "pledged", "backers", "usd pledged"]].info()

<class 'pandas.core.frame.DataFrame'>
Index: 378661 entries, 1000002330 to 999988282
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   goal         378661 non-null  float64
 1   pledged      378661 non-null  float64
 2   backers      378661 non-null  int64  
 3   usd pledged  374864 non-null  float64
dtypes: float64(3), int64(1)
memory usage: 14.4 MB


MinMax Scaling : After Scaling everything lands between 0 and 1 and shape of the distribution is preserved.

Formula : (value - min) / (max - min)

In [20]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
kickstarters_minmax = scaler.fit_transform(kickstarters.loc[:, ["goal", "pledged", "backers", "usd pledged"]])

kickstarters_minmax

array([[9.99990000e-06, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [2.99999900e-04, 1.19032481e-04, 6.83738866e-05, 4.91666589e-06],
       [4.49999900e-04, 1.08166650e-05, 1.36747773e-05, 1.08166650e-05],
       ...,
       [1.49999900e-04, 9.83333178e-07, 4.55825911e-06, 9.83333178e-07],
       [1.49999900e-04, 9.83333178e-06, 2.73495547e-05, 9.83333178e-06],
       [1.99999000e-05, 2.57633293e-05, 7.74904049e-05, 2.57633293e-05]])

Standardization : After Scaling the data is zero centered (i.e) mean = 0 and variance is 1.

Formula : (value - mean) / (standard deviation)

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
kickstarters_standardized = scaler.fit_transform(kickstarters.loc[:, ["goal", "pledged", "backers", "usd pledged"]])

kickstarters_standardized

array([[-0.04062972, -0.10124839, -0.11642345, -0.08948068],
       [-0.01612384, -0.07593363, -0.09988877, -0.08820906],
       [-0.00344839, -0.098948  , -0.11311652, -0.08668311],
       ...,
       [-0.0287993 , -0.10103926, -0.11532114, -0.08922636],
       [-0.0287993 , -0.09915713, -0.10980958, -0.08693744],
       [-0.03978469, -0.09576928, -0.09768414, -0.08281738]])

**When to choose what ?**

* Has extreme outliers?         → Standardization
* Need strict 0-1 range?        → Min-Max
* Data is heavily skewed?       → Log transform first, then either
* Comparing across algorithms?  → Standardization is safer default

Transformations : These are different from normalization, They convert the whole data into a normal distribution. eg : Log Transformation.

The use case of Transformation is when the data is skewed. Log compressed large numbers more than smaller ones.

In [22]:
import numpy as np

In [23]:
kickstarters['goal_log'] = np.log(kickstarters["goal"])

kickstarters[['goal', 'goal_log']]

,goal,goal_log
ID,,
1000002330,1000.0,6.907755
1000003930,30000.0,10.308953
1000004038,45000.0,10.714418
1000007540,5000.0,8.517193
1000011046,19500.0,9.878170
...,...,...
999976400,50000.0,10.819778
999977640,1500.0,7.313220
999986353,15000.0,9.615805


Sometimes there might be values like 0 and log(0) is undefined. So its best practice to use log(1+x) and that can be done by np.log1p()

In [24]:
kickstarters['goal_log1p'] = np.log1p(kickstarters["goal"])

kickstarters[['goal', 'goal_log', 'goal_log1p']]

,goal,goal_log,goal_log1p
ID,,,
1000002330,1000.0,6.907755,6.908755
1000003930,30000.0,10.308953,10.308986
1000004038,45000.0,10.714418,10.714440
1000007540,5000.0,8.517193,8.517393
1000011046,19500.0,9.878170,9.878221
...,...,...,...
999976400,50000.0,10.819778,10.819798
999977640,1500.0,7.313220,7.313887
999986353,15000.0,9.615805,9.615872


# **Parsing Dates**

Python parsing format docs : https://strftime.org/

In [25]:
!kaggle datasets download -d "usgs/earthquake-database"
!unzip earthquake-database.zip

Dataset URL: https://www.kaggle.com/datasets/usgs/earthquake-database
License(s): CC0-1.0
  0% 0.00/590k [00:00<?, ?B/s]
100% 590k/590k [00:00<00:00, 954MB/s]
Archive:  earthquake-database.zip
  inflating: database.csv            


In [26]:
earthquakes = pd.read_csv("database.csv")
earthquakes.head()

,Date,Time,Latitude,Longitude,Type,Depth,Depth Error,Depth Seismic Stations,Magnitude,Magnitude Type,...,Magnitude Seismic Stations,Azimuthal Gap,Horizontal Distance,Horizontal Error,Root Mean Square,ID,Source,Location Source,Magnitude Source,Status
0,01/02/1965,13:44:18,19.246,145.616,Earthquake,131.6,NaN,NaN,6.0,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860706,ISCGEM,ISCGEM,ISCGEM,Automatic
1,01/04/1965,11:29:49,1.863,127.352,Earthquake,80.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860737,ISCGEM,ISCGEM,ISCGEM,Automatic
2,01/05/1965,18:05:58,-20.579,-173.972,Earthquake,20.0,NaN,NaN,6.2,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860762,ISCGEM,ISCGEM,ISCGEM,Automatic
3,01/08/1965,18:49:43,-59.076,-23.557,Earthquake,15.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860856,ISCGEM,ISCGEM,ISCGEM,Automatic
4,01/09/1965,13:32:50,11.938,126.427,Earthquake,15.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860890,ISCGEM,ISCGEM,ISCGEM,Automatic


In [27]:
earthquakes["Date"].dtype

dtype('O')

Most of the entries in the "Date" column follow the same format: "month/day/four-digit year". However, the entry at index 3378 follows a completely different pattern. Run the code cell below to see this.

In [28]:
earthquakes["Date"].str.len().value_counts()

,count
Date,
10,23409
24,3


Those 3 rows with 24 in length are corrupted data.

In [29]:
earthquakes[earthquakes["Date"].str.len() == 24]

,Date,Time,Latitude,Longitude,Type,Depth,Depth Error,Depth Seismic Stations,Magnitude,Magnitude Type,...,Magnitude Seismic Stations,Azimuthal Gap,Horizontal Distance,Horizontal Error,Root Mean Square,ID,Source,Location Source,Magnitude Source,Status
3378,1975-02-23T02:58:41.000Z,1975-02-23T02:58:41.000Z,8.017,124.075,Earthquake,623.0,NaN,NaN,5.6,MB,...,NaN,NaN,NaN,NaN,NaN,USP0000A09,US,US,US,Reviewed
7512,1985-04-28T02:53:41.530Z,1985-04-28T02:53:41.530Z,-32.998,-71.766,Earthquake,33.0,NaN,NaN,5.6,MW,...,NaN,NaN,NaN,NaN,1.30,USP0002E81,US,US,HRV,Reviewed
20650,2011-03-13T02:23:34.520Z,2011-03-13T02:23:34.520Z,36.344,142.344,Earthquake,10.1,13.9,289.0,5.8,MWC,...,NaN,32.3,NaN,NaN,1.06,USP000HWQP,US,US,GCMT,Reviewed


Given all of this information, it's your turn to create a new column "date_parsed" in the earthquakes dataset that has correctly parsed dates in it.

Note: When completing this problem, you are allowed to (but are not required to) amend the entries in the "Date" and "Time" columns. Do not remove any rows from the dataset.

In [30]:
earthquakes["date_parsed"] = pd.to_datetime(earthquakes["Date"], format="%m/%d/%Y", errors="coerce")

errors = "raise" : (This is the default one)

errors = "coerce" : (If its throws an error, then make it a NaT (i.e.) Not a Time)

errors = "ignore" : (If error then ignore and keep the exact value)



Create a Pandas Series day_of_month_earthquakes containing the day of the month from the "date_parsed" column.

In [31]:
earthquakes["date_parsed"].dt.day

,date_parsed
0,2.0
1,4.0
2,5.0
3,8.0
4,9.0
...,...
23407,28.0
23408,28.0
23409,28.0
23410,29.0


# **Character Encoding**

Computers only understand binary (0s and 1s), so encoding is a ruleset that maps binary bytes → human readable characters.
When you read a file with the wrong encoding, you get garbled text called mojibake:

æ–‡å—åŒ– ← this is mojibake

In [32]:
!kaggle datasets download -d "kwullum/fatal-police-shootings-in-the-us"
!unzip fatal-police-shootings-in-the-us.zip

Dataset URL: https://www.kaggle.com/datasets/kwullum/fatal-police-shootings-in-the-us
License(s): CC-BY-NC-SA-4.0
  0% 0.00/1.06M [00:00<?, ?B/s]
100% 1.06M/1.06M [00:00<00:00, 136MB/s]
Archive:  fatal-police-shootings-in-the-us.zip
  inflating: MedianHouseholdIncome2015.csv  
  inflating: PercentOver25CompletedHighSchool.csv  
  inflating: PercentagePeopleBelowPovertyLevel.csv  
  inflating: PoliceKillingsUS.csv    
  inflating: ShareRaceByCity.csv     


In [33]:
import charset_normalizer

In [34]:
with open("PoliceKillingsUS.csv", 'rb') as f:
    result = charset_normalizer.detect(f.read())

print(result)

{'encoding': 'windows-1250', 'language': 'English', 'confidence': 1.0}


We could use file handling + charset_normalizer to detect how the dataset is encoded and then proceed further to read the dataset.

In [35]:
police_killings = pd.read_csv("PoliceKillingsUS.csv", encoding="windows-1250")

# **Handling Inconsistent Data**

In [36]:
!kaggle datasets download -d "alexisbcook/pakistan-intellectual-capital"
!unzip pakistan-intellectual-capital.zip

Dataset URL: https://www.kaggle.com/datasets/alexisbcook/pakistan-intellectual-capital
License(s): unknown
  0% 0.00/47.8k [00:00<?, ?B/s]
100% 47.8k/47.8k [00:00<00:00, 91.5MB/s]
Archive:  pakistan-intellectual-capital.zip
  inflating: pakistan_intellectual_capital.csv  


In [38]:
professors = pd.read_csv("pakistan_intellectual_capital.csv")
professors.head()

,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN


In [47]:
professors["Country"].unique()

array(['Thailand', 'Pakistan', 'germany', 'Austria', 'Australia', 'UK',
       'China', 'France', 'USofA', 'SouthKorea', 'Malaysia', 'Sweden',
       'Italy', 'Canada', 'Norway', 'Ireland', 'New Zealand', 'Urbana',
       'Portugal', 'Russian Federation', 'USA', 'Finland', ' USA',
       'Netherland', ' Germany', ' Sweden', ' New Zealand', 'Greece',
       'Turkey', 'South Korea', 'Macau', 'Singapore', 'Spain', 'Japan',
       'HongKong', 'Saudi Arabia', 'Mauritius', 'Scotland'], dtype=object)

In [50]:
!pip install fuzzywuzzy

Converting the "Country" column to all lowercase and stripping the whitespaces on the front end back of the string.

In [57]:
professors["Country"] = professors["Country"].str.lower()
professors["Country"] = professors["Country"].str.strip()

Handling Incosistent Country Names

In [59]:
professors["Country"].unique()

array(['thailand', 'pakistan', 'germany', 'austria', 'australia', 'uk',
       'china', 'france', 'usofa', 'southkorea', 'malaysia', 'sweden',
       'italy', 'canada', 'norway', 'ireland', 'new zealand', 'urbana',
       'portugal', 'russian federation', 'usa', 'finland', 'netherland',
       'greece', 'turkey', 'south korea', 'macau', 'singapore', 'spain',
       'japan', 'hongkong', 'saudi arabia', 'mauritius', 'scotland'],
      dtype=object)

Fuzzzywuzzy : https://pypi.org/project/fuzzywuzzy/

It's a library that measures the similarity between tokens(words) using a promixity metric of letter differences. Thus, helps in handling inconsistent words in the dataset.

In [52]:
from fuzzywuzzy import process, fuzz

In [62]:
fuzz.ratio("Mumbai", "mumbay")

67

In [63]:
fuzz.partial_ratio("South Korea", "Korea")

100

In [64]:
fuzz.token_sort_ratio("South Korea", "Korea South")

100

In [66]:
fuzz.token_set_ratio("south korea", "south republic of korea")

100

fuzz.ratio(str1, str2) : Strict and compares the letter differences to caluclate the score.

fuzz.partial_ratio(str1, str2) : Checks if one word is the subset of another.

fuzz.token_sort_ratio(str1, str2) : Order doesn't matter.

fuzz.token_set_ratio(str1, str2) : Splits into tokens and compares the subset.

In [67]:
correct = ["pakistan", "usa", "south korea", "netherlands", "hong kong"]

process.extractOne("usofa", correct)

('usa', 75)

process.extractOne(str1, iter) : Check the similarity between the str1 and all the str in the iter and then returns the best match along with the similarity score.